# DiDAQt FABRIC Experiment

This notebook creates a FABRIC testbed artifact that demonstrates the DiDAQt
fault-detection framework for DAQ networks.

**Topology:**
- 1 Sender node (ConnectX-6, 2 ports) — runs 10 sender instances
- 1 Receiver node (ConnectX-6, 2 ports + 2 NIC_Basic) — runs 2 receiver instances
- 1 Controller node (NIC_Basic) — runs the heartbeat monitor
- 1 Tofino P4 switch — L2 MAC forwarding with VLAN rewriting

**Data path (in-band):** Sender ports ↔ Tofino ↔ Receiver ports (VLAN-tagged)
- Cross-site links use **L2PTP**; same-site links use **L2Bridge**.

**Control path (out-of-band):** Receiver NICs → Controller (FABNetv4 / L3)

## 1. Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()

In [ ]:
fablib.list_sites(filter_function=lambda x: x["p4-switch_available"] > 0 and x["nic_connectx_6_available"] > 1, pretty_names=False)

In [ ]:
import os
from concurrent import futures

# ---------- Configuration ----------
# Each component can be placed on a separate FABRIC site.
# Cross-site in-band links use L2PTP; same-site links use L2Bridge.
# Run fablib.list_sites() to find sites with Tofino switches and ConnectX-6 NICs.

SENDER_SITE     = 'UTAH'
RECEIVER_SITE   = 'UTAH'
SWITCH_SITE     = 'UTAH'
CONTROLLER_SITE = 'UTAH'

SLICE_NAME = 'didaqt-experiment'
IMAGE      = 'default_ubuntu_22'

# SDE environment command on FABRIC Tofino nodes (nix-shell wrapper).
# Adjust the version if your site uses a different SDE release.
SDE_ENV = 'sde-env-9.13.3'

# Paths relative to this notebook (artifact/)
REPO = os.path.abspath('..')
REMOTE_DIR = '/home/ubuntu/didaqt'

## 2. Create Topology

In [ ]:
slice = fablib.new_slice(name=SLICE_NAME)

# ---- Nodes ----
sender_node = slice.add_node(name='sender', site=SENDER_SITE,
                             cores=8, ram=32, disk=20, image=IMAGE)
receiver_node = slice.add_node(name='receiver', site=RECEIVER_SITE,
                               cores=8, ram=32, disk=20, image=IMAGE)
controller_node = slice.add_node(name='controller', site=CONTROLLER_SITE,
                                 cores=4, ram=8, disk=20, image=IMAGE)

# ---- P4 Switch ----
p4_switch = slice.add_switch(name='p4_switch', site=SWITCH_SITE)

# ---- In-band NICs (ConnectX-6, dual-port 100G) ----
sender_nic = sender_node.add_component(model='NIC_ConnectX_6', name='sender_nic')
rx_nic     = receiver_node.add_component(model='NIC_ConnectX_6', name='rx_nic')

sender_ifaces = sender_nic.get_interfaces()
rx_ifaces     = rx_nic.get_interfaces()
sw_ifaces     = p4_switch.get_interfaces()

print(f'Sender NIC interfaces:   {[i.get_name() for i in sender_ifaces]}')
print(f'Receiver NIC interfaces: {[i.get_name() for i in rx_ifaces]}')
print(f'Switch interfaces:       {[i.get_name() for i in sw_ifaces]}')

# ---- In-band L2 networks ----
# Use L2PTP when the two endpoints are on different sites, L2Bridge when same-site.
# Create networks first, then add interfaces (following FABRIC switch pattern).

def l2_type(site_a, site_b):
    return 'L2PTP' if site_a != site_b else 'L2Bridge'

l2_type_sender_sw = l2_type(SENDER_SITE, SWITCH_SITE)
l2_type_rx_sw     = l2_type(RECEIVER_SITE, SWITCH_SITE)

print(f'\nSender  <-> Switch link type: {l2_type_sender_sw}')
print(f'Receiver <-> Switch link type: {l2_type_rx_sw}')

# Create the four L2 networks.
net_s0 = slice.add_l2network(name='net-s0-sw', type=l2_type_sender_sw)
net_s1 = slice.add_l2network(name='net-s1-sw', type=l2_type_sender_sw)
net_r0 = slice.add_l2network(name='net-r0-sw', type=l2_type_rx_sw)
net_r1 = slice.add_l2network(name='net-r1-sw', type=l2_type_rx_sw)

# Add node interfaces (set_mode='config' for post-boot configuration).
for iface, net in [(sender_ifaces[0], net_s0), (sender_ifaces[1], net_s1),
                   (rx_ifaces[0], net_r0), (rx_ifaces[1], net_r1)]:
    iface.set_mode('config')
    net.add_interface(iface)

# Add switch interfaces.
net_s0.add_interface(sw_ifaces[0])
net_s1.add_interface(sw_ifaces[1])
net_r0.add_interface(sw_ifaces[2])
net_r1.add_interface(sw_ifaces[3])

# ---- Out-of-band NICs (NIC_Basic for heartbeat L3 network) ----
rx_ctrl_nic0 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl0')
rx_ctrl_nic1 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl1')
ctrl_nic     = controller_node.add_component(model='NIC_Basic', name='ctrl_nic')

ctrl_net = slice.add_l3network(name='ctrl-net', interfaces=[
    rx_ctrl_nic0.get_interfaces()[0],
    rx_ctrl_nic1.get_interfaces()[0],
    ctrl_nic.get_interfaces()[0]
], type='IPv4')

print('\nTopology defined.')
slice.show()

## 3. Submit Slice

In [ ]:
slice.submit()
print('Slice submitted and ready.')

## 4. Gather Topology Information

After the slice is provisioned, query the actual OS interface names,
MAC addresses, VLAN IDs, and L3 IP addresses.

In [ ]:
slice = fablib.get_slice(name=SLICE_NAME)

sender_node     = slice.get_node('sender')
receiver_node   = slice.get_node('receiver')
controller_node = slice.get_node('controller')
p4_switch       = slice.get_node('p4_switch')

def vlan_or_zero(iface):
    """Return the VLAN ID as an int, or 0 if the link has no VLAN (L2Bridge)."""
    v = iface.get_vlan()
    return int(v) if v else 0

# ---- Sender in-band interfaces ----
s_iface0 = sender_node.get_interface(network_name='net-s0-sw')
s_iface1 = sender_node.get_interface(network_name='net-s1-sw')

s0_os   = s_iface0.get_physical_os_interface_name()
s0_mac  = s_iface0.get_mac()
s0_vlan = vlan_or_zero(s_iface0)
s1_os   = s_iface1.get_physical_os_interface_name()
s1_mac  = s_iface1.get_mac()
s1_vlan = vlan_or_zero(s_iface1)

print(f'Sender port 0: iface={s0_os}  mac={s0_mac}  vlan={s0_vlan}')
print(f'Sender port 1: iface={s1_os}  mac={s1_mac}  vlan={s1_vlan}')

# ---- Receiver in-band interfaces ----
r_iface0 = receiver_node.get_interface(network_name='net-r0-sw')
r_iface1 = receiver_node.get_interface(network_name='net-r1-sw')

r0_os   = r_iface0.get_physical_os_interface_name()
r0_mac  = r_iface0.get_mac()
r0_vlan = vlan_or_zero(r_iface0)
r1_os   = r_iface1.get_physical_os_interface_name()
r1_mac  = r_iface1.get_mac()
r1_vlan = vlan_or_zero(r_iface1)

print(f'\nReceiver port 0: iface={r0_os}  mac={r0_mac}  vlan={r0_vlan}')
print(f'Receiver port 1: iface={r1_os}  mac={r1_mac}  vlan={r1_vlan}')

if s0_vlan == 0:
    print('\nNote: VLAN IDs are 0 (L2Bridge / same-site). Frames will be untagged.')
else:
    print(f'\nVLAN tagging is active (L2PTP / cross-site).')

# ---- Controller L3 interface ----
ctrl_iface = controller_node.get_interface(network_name='ctrl-net')
ctrl_ip = ctrl_iface.get_ip_addr()

# The receiver has two NIC_Basic on ctrl-net; get both IPs.
rx_ctrl_ifaces = [i for i in receiver_node.get_interfaces()
                  if i.get_network() and i.get_network().get_name() == 'ctrl-net']
rx_ctrl_ips = [i.get_ip_addr() for i in rx_ctrl_ifaces]

print(f'\nController IP: {ctrl_ip}')
print(f'Receiver ctrl IPs: {rx_ctrl_ips}')

## 5. Install Dependencies

In [ ]:
install_cmd = ('sudo apt-get update -qq && '
               'sudo apt-get install -y -qq build-essential ethtool libyaml-dev')

from concurrent.futures import ThreadPoolExecutor

def install_on(node):
    name = node.get_name()
    print(f'Installing on {name}...')
    stdout, stderr = node.execute(install_cmd, quiet=True)
    print(f'  {name}: done')
    return stdout, stderr

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(install_on, n)
               for n in [sender_node, receiver_node, controller_node]]
    for f in futures:
        f.result()

print('All dependencies installed.')

## 6. Upload Source Code

In [ ]:
jobs = []

for node in [sender_node, receiver_node, controller_node]:
    jobs.append(node.upload_directory_thread(REPO, f'{REMOTE_DIR}'))

# Switch — only the P4 program
p4_switch.execute(f'mkdir -p examples/p4', quiet=True)
p4_switch.upload_file(
    os.path.join(REPO, 'examples/p4/l2_forward.p4'),
    f'examples/p4/l2_forward.p4'
)

for job in futures.as_completed(jobs):
    continue

print('Upload complete.')

## 7. Compile

In [ ]:
build_dir = f'{REMOTE_DIR}/build'

jobs = []
for node in [sender_node, receiver_node, controller_node]:
    jobs.append(node.execute_thread(f'cd {REMOTE_DIR} && make examples'))

for job in futures.as_completed(jobs):
    continue

print('All binaries compiled.')

## 8. Configure P4 Switch

Compile the P4 program on the Tofino, upload the port-enable script,
forwarding rules, and switch agent.  Then print the manual commands
to run on the switch via SSH.

In [ ]:
import os

# Switch uses /home/fabric, not /home/ubuntu — use relative paths.
P4_SRC = 'examples/p4/l2_forward.p4'
P4_PROG = 'l2_forward'

# Regex that matches the nix-shell prompt.
NIX_PROMPT = r'.*nix-shell.*'

# Regex that matches the normal shell prompt.
SW_PROMPT = r'.*fabric@p4switch.*'

# ---- Step 1: Compile P4 program ----
print('Compiling P4 program on switch...')
stdout, stderr = p4_switch.execute(command=[
    (SDE_ENV,                          NIX_PROMPT, 30),
    (f'p4_build.sh ~/{P4_SRC}',       NIX_PROMPT, 120),
    ('exit',                           SW_PROMPT,    10),
])
print('  P4 compilation done.')

# Verify build output
stdout, stderr = p4_switch.execute(command=[
    (SDE_ENV,                                                    NIX_PROMPT, 10),
    (f'ls ~/.bf-sde/*/build/{P4_PROG}/tofino/pipe/ 2>/dev/null', NIX_PROMPT, 10),
    ('exit',                                                     SW_PROMPT,   10),
])
print(f'  Build artifacts: {stdout.strip()}')

# ---- Step 2: Get logical port names ----
sw_iface_s0 = p4_switch.get_interface(network_name='net-s0-sw')
sw_iface_s1 = p4_switch.get_interface(network_name='net-s1-sw')
sw_iface_r0 = p4_switch.get_interface(network_name='net-r0-sw')
sw_iface_r1 = p4_switch.get_interface(network_name='net-r1-sw')

sw_port_s0 = sw_iface_s0.get_device_name()
sw_port_s1 = sw_iface_s1.get_device_name()
sw_port_r0 = sw_iface_r0.get_device_name()
sw_port_r1 = sw_iface_r1.get_device_name()

print(f'  Switch logical ports: s0={sw_port_s0} s1={sw_port_s1} '
      f'r0={sw_port_r0} r1={sw_port_r1}')

# ---- Step 3: Generate and upload port-enable ucli script ----
port_script = f"""pm port-add {sw_port_s0}/- 100G NONE
pm port-add {sw_port_s1}/- 100G NONE
pm port-add {sw_port_r0}/- 100G NONE
pm port-add {sw_port_r1}/- 100G NONE
pm port-enb {sw_port_s0}/-
pm port-enb {sw_port_s1}/-
pm port-enb {sw_port_r0}/-
pm port-enb {sw_port_r1}/-
pm show
"""

import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(port_script)
    local_port_script = f.name
p4_switch.upload_file(local_port_script, '/tmp/enable_ports.txt')
os.unlink(local_port_script)
print('  enable_ports.txt uploaded')

# ---- Step 4: Upload switch agent ----
p4_switch.upload_file(
    os.path.join(REPO, 'artifact/switch_agent.py'),
    '/tmp/switch_agent.py'
)
print('  switch_agent.py uploaded')

print('\nP4 compilation and script upload complete.')

In [ ]:
# The setup_rules.py script was already generated and uploaded.
# It was executed inside the bfshell session in the previous cell.
# This cell just regenerates it if you need to re-upload.

def mac_to_int(mac_str):
    """Convert 'AA:BB:CC:DD:EE:FF' to integer."""
    return int(mac_str.replace(':', ''), 16)

sw_vlan_r0 = vlan_or_zero(sw_iface_r0)
sw_vlan_r1 = vlan_or_zero(sw_iface_r1)

bfrt_script = f'''
logical_ports = {{
    'r0': '{sw_port_r0}',
    'r1': '{sw_port_r1}',
}}

port_dump = bfrt.port.port.dump(return_ents=True)
port_map = {{}}
if port_dump:
    for ent in port_dump:
        dp = ent.key.get('$DEV_PORT') or ent.key.get(b'$DEV_PORT')
        pname = ent.data.get('$PORT_NAME') or ent.data.get(b'$PORT_NAME', '')
        if isinstance(pname, bytes):
            pname = pname.decode()
        pname = str(pname)
        connector = pname.split('/')[0] if '/' in pname else pname
        port_map[connector] = dp

print(f"Port mapping: {{port_map}}")

def resolve(name, logical):
    dp = port_map.get(logical)
    if dp is None:
        print(f"WARNING: could not resolve logical port {{logical}} for {{name}}")
        return None
    print(f"  {{name}}: logical={{logical}} -> dev_port={{dp}}")
    return dp

dev_r0 = resolve('receiver0', logical_ports['r0'])
dev_r1 = resolve('receiver1', logical_ports['r1'])

l2_fwd = bfrt.l2_forward.pipe.Ingress.l2_forward

if dev_r0 is not None:
    l2_fwd.add_with_forward(
        dst_addr={mac_to_int(r0_mac)},
        port=dev_r0,
        vlan_id={sw_vlan_r0}
    )
    print("Rule: dst_mac={r0_mac} -> port={{dev_r0}} vlan={sw_vlan_r0}")

if dev_r1 is not None:
    l2_fwd.add_with_forward(
        dst_addr={mac_to_int(r1_mac)},
        port=dev_r1,
        vlan_id={sw_vlan_r1}
    )
    print("Rule: dst_mac={r1_mac} -> port={{dev_r1}} vlan={sw_vlan_r1}")

bfrt.complete_operations()
l2_fwd.dump(table=True)
'''

import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(bfrt_script)
    local_script = f.name

p4_switch.upload_file(local_script, '/tmp/setup_rules.py')
os.unlink(local_script)
print('setup_rules.py uploaded to switch (executed by the bfshell session).')

## 8c. Switch Manual Commands and Connectivity Test

The switch must be configured manually via SSH because the
`execute_thread` interactive prompt matching is fragile for
long-running processes like `bf_switchd`.

Run these commands in an SSH session on the switch, then use
the test below to verify the agent is reachable.

In [ ]:
# ---- Print manual switch commands ----
sw_ssh = p4_switch.get_ssh_command()
sw_mgmt_ip = p4_switch.get_management_ip()

print('='*70)
print('SWITCH SETUP: Run these commands in an SSH session on the switch')
print('='*70)
print(f'\n  {sw_ssh}\n')
print(f'''
# 1. Enter the SDE environment
{SDE_ENV}

# 2. Load kernel modules
sudo $SDE_INSTALL/bin/./bf_kdrv_mod_load $SDE_INSTALL

# 3. Start bf_switchd (stays in foreground — use a tmux/screen session)
run_switchd.sh -p l2_forward

# 4. In the bfshell> prompt, enable ports:
ucli
source /tmp/enable_ports.txt
exit

# 5. Install forwarding rules:
bfrt_python /tmp/setup_rules.py

# 6. Start the switch agent (blocks — handles controller commands):
bfrt_python /tmp/switch_agent.py

# To stop the agent later: send QUIT via ICMPv6, or from another
# SSH session on the switch: touch /tmp/switch_agent_stop
''')

# ---- Connectivity test ----
print('='*70)
print('CONNECTIVITY TEST (run after completing the switch setup above)')
print('='*70)

print(f'\nSwitch management IP: {sw_mgmt_ip}')

print(f'\nPing test: controller -> switch ({sw_mgmt_ip})...')
stdout, stderr = controller_node.execute(
    f'ping -c 3 -W 2 {sw_mgmt_ip}', quiet=True
)
print(stdout)

# ICMPv6 agent test — must run with sudo (raw sockets require root)
print(f'ICMPv6 agent test: controller -> switch ({sw_mgmt_ip})...')
icmp_test_script = f'''
import socket, struct, sys
ICMP6_REQ, ICMP6_REP = 128, 129
ID = 0xDDAA
try:
    sock = socket.socket(socket.AF_INET6, socket.SOCK_RAW, socket.IPPROTO_ICMPV6)
except PermissionError:
    print("ERROR: raw socket requires root", file=sys.stderr)
    sys.exit(1)
sock.settimeout(3)
pkt = struct.pack('!BBHHH', ICMP6_REQ, 0, 0, ID, 1) + b'PING'
sock.sendto(pkt, ('{sw_mgmt_ip}', 0, 0, 0))
for _ in range(5):
    try:
        data, addr = sock.recvfrom(256)
    except socket.timeout:
        print('TIMEOUT')
        break
    if len(data) < 8: continue
    if data[0] != ICMP6_REP: continue
    rid = struct.unpack('!H', data[4:6])[0]
    if rid != ID: continue
    payload = data[8:].decode(errors='replace').strip()
    if payload.startswith('PONG'):
        print(f'PONG from {{addr[0]}}')
        break
else:
    print('NO AGENT REPLY')
sock.close()
'''

# Write script to a file and run with sudo (avoids shell quoting issues)
controller_node.execute('cat > /tmp/icmp_test.py << \'PYEOF\'\n' + icmp_test_script + '\nPYEOF', quiet=True)
stdout, stderr = controller_node.execute('sudo python3 /tmp/icmp_test.py', quiet=True)
response = stdout.strip()
print(f'  Response: {response}')

if stderr.strip():
    print(f'  stderr: {stderr.strip()}')

if 'PONG' in response:
    print('  Switch agent is reachable via ICMPv6!')
elif 'TIMEOUT' in response:
    print('  Timed out — the switch agent may not be running yet.')
    print('  Complete the switch setup steps above and re-run this cell.')
else:
    print('  Agent not responding — complete the switch setup steps above first.')

# ---- Save switch address for run commands ----
switch_agent_addr = str(sw_mgmt_ip)
print(f'\nSwitch agent address for controller: {switch_agent_addr}')

## 8d. Generate Topology YAML

Build the DiDAQt topology description from the discovered switch
ports and receiver IDs.  This YAML is fed to the controller for
static reachability analysis and failover state tracking.

In [ ]:
# Generate topology YAML for the 10-sender / 2-receiver / 1-switch example.
# Sender port on the switch: sw_port_s0 (logical port name).
# Receiver ports on the switch: sw_port_r0, sw_port_r1.

NUM_SENDERS = 10

def sender_ic_block(senders, receiver):
    """Build the initial_connections sub-block for a list of senders."""
    lines = []
    for idx, s in enumerate(senders, 1):
        lines.append(f'        {idx}:')
        lines.append(f'          sender: {s}')
        lines.append(f'          receiver: {receiver}')
    return '\n'.join(lines)

sender_names = [f'S{i}' for i in range(1, NUM_SENDERS + 1)]
ic_block = sender_ic_block(sender_names, 'R0')

# ---- Sender entries ----
sender_entries = []
for i in range(1, NUM_SENDERS + 1):
    sender_entries.append(f'''- name: S{i}
  type: sender
  sender_id: {i}
  sender_id_bytes: 26
  max_bandwidth: 10G
  initial_receiver: R0
  group_id: 0
  connections:
    1:
      other_node: SW1
      other_port: {sw_port_s0}
      max_bandwidth: 100G
      initial_connections:
        1:
          sender: S{i}
          receiver: R0''')

# ---- Switch entry ----
switch_entry = f'''- name: SW1
  type: switch
  switch_type_group: tofino2
  connections:
    {sw_port_s0}:
      other_node: S1
      other_port: 1
      max_bandwidth: 100G
      initial_connections:
{ic_block}
    {sw_port_r0}:
      other_node: R0
      other_port: 1
      max_bandwidth: 100G
      initial_connections:
{ic_block}
    {sw_port_r1}:
      other_node: R1
      other_port: 1
      max_bandwidth: 100G'''

# ---- Receiver entries ----
r0_entry = f'''- name: R0
  type: receiver
  receiver_id: 0
  connections:
    1:
      other_node: SW1
      other_port: {sw_port_r0}
      max_bandwidth: 100G
      initial_connections:
{ic_block}'''

r1_entry = f'''- name: R1
  type: receiver
  receiver_id: 1
  connections:
    1:
      other_node: SW1
      other_port: {sw_port_r1}
      max_bandwidth: 100G'''

topology_yaml = '\n\n'.join(sender_entries + [switch_entry, r0_entry, r1_entry]) + '\n'

print('--- Generated topology.yaml ---')
print(topology_yaml[:500] + '...\n')

# Write locally and upload to the controller node
topo_local = '/tmp/didaqt_topology.yaml'
with open(topo_local, 'w') as f:
    f.write(topology_yaml)

controller_node.upload_file(topo_local, f'{REMOTE_DIR}/topology.yaml')
print(f'Topology YAML uploaded to controller ({REMOTE_DIR}/topology.yaml)')

## 9. Configure Node Interfaces

Bring up the in-band (L2) interfaces.  When VLAN tagging is active
(L2PTP / cross-site), VLAN offloading is disabled so the sender and
receiver handle tags in software.
The L3 (FABNetv4) interfaces are auto-configured by FABRIC.

In [ ]:
vlan_active = s0_vlan > 0  # True when L2PTP (cross-site) is in use

# ---- Sender: bring up interfaces ----
for iface, name in [(s0_os, 'sender port 0'), (s1_os, 'sender port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    sender_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- Receiver: bring up in-band interfaces ----
for iface, name in [(r0_os, 'receiver port 0'), (r1_os, 'receiver port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    receiver_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- L3 interfaces are auto-configured by FABRIC ----
# Verify connectivity
print(f'\nVerifying L3 connectivity: receiver -> controller ({ctrl_ip})...')
stdout, stderr = receiver_node.execute(
    f'ping -c 2 -W 2 {ctrl_ip}',
    quiet=True
)
print(stdout)
print('Interface configuration complete.')

## 10. SSH Access

Use these commands to SSH into each node from your local terminal.

In [ ]:
print('='*70)
print('SSH Commands')
print('='*70)
for node in [sender_node, receiver_node, controller_node, p4_switch]:
    name = node.get_name()
    ssh_cmd = node.get_ssh_command()
    print(f'\n--- {name} ---')
    print(f'  {ssh_cmd}')
print()

## 11. Run the Experiment

The cells below print the exact commands to run on each node.
Open SSH sessions to each node (Section 10) and paste these commands.

**Start order:**
1. Controller (heartbeat monitor)
2. Receiver (both instances)
3. Sender (10 instances)

In [ ]:
HB_PORT = 9000

# Build the sender VLAN argument: only pass it when vlan_id > 0.
sender_vlan_arg = f' {s0_vlan}' if s0_vlan else ''

print('='*70)
print('STEP 1: Start DiDAQt controller on CONTROLLER')
print('='*70)
print(f'''
cd {REMOTE_DIR}

# With live switch updates via ICMPv6 (measures failover RTT):
sudo ./build/controller topology.yaml {HB_PORT} {switch_agent_addr}

# Or log-only mode (no switch agent):
# sudo ./build/controller topology.yaml {HB_PORT}

# Or raw heartbeat monitoring (no failover logic):
# sudo ./build/heartbeat_monitor {HB_PORT}
''')
print('  The controller sends switch commands via ICMPv6 echo request/reply')
print('  to the agent running inside bfshell on the Tofino.')
print()

print('='*70)
print('STEP 2: Start receiver instances on RECEIVER')
print('='*70)
print(f'''
cd {REMOTE_DIR}

# Receiver instance 0: listens on port 0, heartbeats to controller
sudo ./build/receiver {r0_os} 0 {ctrl_ip} {HB_PORT} &

# Receiver instance 1: listens on port 1, heartbeats to controller
sudo ./build/receiver {r1_os} 1 {ctrl_ip} {HB_PORT} &
''')

print('='*70)
print('STEP 3: Start 10 sender instances on SENDER')
print('='*70)
print(f'''
cd {REMOTE_DIR}

# All 10 senders target receiver port 0 via sender port 0.
for i in $(seq 1 10); do
    sudo ./build/sender {s0_os} {r0_mac} $i{sender_vlan_arg} &
done

# To stop all senders:
# sudo killall sender
''')

if sender_vlan_arg:
    print(f'(Frames are VLAN-tagged with VID {s0_vlan})')
else:
    print('(Frames are untagged — L2Bridge / same-site)')

print()
print('='*70)
print('STEP 4 (optional): Test failover with faulty sender')
print('='*70)
print(f'''
# Kill one normal sender and restart it with the -f (faulty) flag:
sudo kill %1   # kill sender 1
sudo ./build/sender -f {s0_os} {r0_mac} 1{sender_vlan_arg} &
''')

print('='*70)
print('EXPECTED OUTPUT')
print('='*70)
print(f'''
On the controller:

  ICMPv6 switch agent: {switch_agent_addr}
  Switch agent PING OK: PONG (rtt=500 us)
  Loading topology: topology.yaml
  Initial paths (20 total):
    [0] sender=S1 -> receiver=R0  USED
    [1] sender=S1 -> receiver=R1  AVAILABLE
    ...

  [HB #1] receiver_id=0 senders=10  proc=15 us
  [HB #2] receiver_id=1 senders=0   proc=5 us

When a faulty sender triggers failover:

  [HB #42] receiver_id=0 senders=9  proc=2100 us
    SWITCH UPDATE: sender_id=1  (1,3) -> (1,4)  rtt=1800 us  resp=OK 450

The rtt is the ICMPv6 round-trip to the switch agent (MTTR metric).
''')

## 12. Cleanup

Delete the slice when you are done with the experiment.

In [ ]:
# Uncomment to delete:
# slice.delete()
# print('Slice deleted.')